# Dominion Energy South Carolina: project ingestion
Extract the 2026?2030 SCRTP project descriptions into one row per project. Keep the original PDF, retain page-level provenance, review source issues, and save cleaned CSV/JSON files.

Select the project's `.venv` interpreter/kernel. If dependencies are missing, run `%pip install -e "..[notebooks]"` from this notebook's directory and restart the kernel. The cached PDF allows subsequent runs without network access.

In [1]:
import hashlib
import json
import re
from datetime import datetime

import pandas as pd
import pymupdf
import requests
from IPython.display import display

from gridlock.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR

URL = "https://www.scrtp.com/assets/pdfs/home/2026-2030-2million-and-above-project-descriptions.pdf"
PDF_PATH = RAW_DATA_DIR / "dominion" / "scrtp_projects.pdf"
OUTPUT_DIR = PROCESSED_DATA_DIR / "dominion"
pd.set_option("display.max_colwidth", 100)

## 1. Load the original document
Download only when the cached file is absent. Validate the response before saving; never overwrite the original during a normal rerun.

In [2]:
if not PDF_PATH.exists():
    response = requests.get(URL, timeout=60)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF-"):
        raise ValueError("The source returned something other than a PDF.")
    with pymupdf.open(stream=response.content, filetype="pdf") as candidate:
        if not len(candidate):
            raise ValueError("The downloaded PDF has no pages.")
    PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    with PDF_PATH.open("xb") as file:
        file.write(response.content)

source_sha256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
with pymupdf.open(PDF_PATH) as document:
    pages = [{"source_page": i, "raw_text": page.get_text("text")}
             for i, page in enumerate(document, start=1)]
df_pages = pd.DataFrame(pages)
print(f"Loaded {len(df_pages)} pages from {PDF_PATH}")
print(df_pages.iloc[0]["raw_text"])

Loaded 54 pages from C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\dominion\scrtp_projects.pdf
Project 1 of 54 
 
Dominion Energy South Carolina 
Planned Transmission Projects $2M and above Total 
5 Year Budget 
 
 
 
 
Summerville 115kV Loop: Rebuild 
 
Project ID 
06810 H 
 
Project Description 
Replacing several wood pole sections on the Summerville 115 kV Loop that have reached the end of life. 
 
Project Need 
This project is required to address end of life and maintenance issues. 
 
Project Status 
In Progress 
 
Planned In-Service Date 
04/31/26 
 
Estimated Project Cost 
Previous 
2026 
2027 
2028 
2029 
2030 
Total 
$13,675,885 
$2,100,000 
 
$0 
 
$0 
 
$0 
$0 
$15,775,885 
 
 
 
 
 



## 2. Parse the labeled project fields
The source has one project per page. Extract fields using section labels, including multiline titles and descriptions. Preserve project IDs as strings (including leading zeros).

For cost tables, use the published **Total**, retain each annual amount, and flag disagreements with their sum. Customer-financed projects use the explicitly stated estimate. These are source-reported estimates, not verified lifetime costs.

Set `state="SC"` here because this is the South Carolina source dataset. All later exports retain it.

In [3]:
from gridlock.geolocation import get_action_type, extract_actions

def compact(value):
    return re.sub(r"\s+", " ", value).strip()


def section(text, start, end=None):
    pattern = rf"{re.escape(start)}\s*(.*?)" + (rf"\s*{re.escape(end)}" if end else r"\s*\Z")
    match = re.search(pattern, text, flags=re.S)
    if not match:
        raise ValueError(f"Missing section: {start}")
    return compact(match.group(1))


def parse_dates(raw):
    tokens = re.findall(r"\b\d{1,2}/\d{1,2}/(?:\d{4}|\d{2})\b", raw)
    dates, issues = [], []
    for token in tokens:
        fmt = "%m/%d/%Y" if len(token.rsplit("/", 1)[1]) == 4 else "%m/%d/%y"
        try:
            dates.append(datetime.strptime(token, fmt).date().isoformat())
        except ValueError:
            issues.append(f"invalid_in_service_date:{token}")
    if not tokens:
        issues.append("missing_in_service_date")
    if len(tokens) > 1:
        issues.append("multiple_in_service_dates")
    # A phased or invalid date requires review; do not silently choose a phase.
    single = dates[0] if len(tokens) == 1 and len(dates) == 1 else None
    return single, dates, issues


BUDGET_COLUMNS = ["cost_previous", "cost_2026", "cost_2027", "cost_2028", "cost_2029", "cost_2030"]


def parse_cost(raw):
    budget = dict.fromkeys(BUDGET_COLUMNS)
    issues = []
    if "Previous" in raw:
        match = re.fullmatch(r"Previous\s+2026\s+2027\s+2028\s+2029\s+2030\s+Total\s+(.+)", raw)
        if not match:
            raise ValueError(f"Unexpected cost table: {raw}")
        tokens = re.findall(r"\$?\d[\d,]*(?:\.\d+)?", match.group(1))
        if len(tokens) != 7:
            raise ValueError(f"Expected seven budget values, found {len(tokens)}")
        for token in tokens:
            if not re.fullmatch(r"\$?(?:\d+|\d{1,3}(?:,\d{3})+)", token):
                issues.append("nonstandard_cost_format")
        amounts = [int(token.replace("$", "").replace(",", "")) for token in tokens]
        budget.update(zip(BUDGET_COLUMNS, amounts[:-1]))
        total = amounts[-1]
        difference = total - sum(amounts[:-1])
        if difference:
            issues.append("cost_total_mismatch")
        return total, "published_budget_total", budget, difference, issues
    match = re.search(r"Estimated cost of\s+\$([\d,]+)", raw, flags=re.I)
    if not match:
        raise ValueError(f"Unrecognized narrative cost: {raw}")
    return int(match.group(1).replace(",", "")), "customer_financed_estimate", budget, None, issues


def parse_project(page):
    text = page["raw_text"]
    header = re.search(r"Project\s+(\d+)\s+of\s+(\d+)", text)
    if not header:
        raise ValueError(f"Page {page['source_page']} has no project header")
    name = section(text, "5 Year Budget", "Project ID")
    date_raw = section(text, "Planned In-Service Date", "Estimated Project Cost")
    date, dates, issues = parse_dates(date_raw)
    cost_raw = section(text, "Estimated Project Cost")
    total, basis, budget, difference, cost_issues = parse_cost(cost_raw)
    issues.extend(cost_issues)
    if "\ufffd" in text:
        issues.append("pdf_text_replacement_character")
    return {
        "utility": "Dominion Energy South Carolina",
        "state": "SC",
        "project_name": name,
        "action_type": get_action_type(name),
        "detected_actions": ";".join(extract_actions(name)),
        "project_id": section(text, "Project ID", "Project Description"),
        "description": section(text, "Project Description", "Project Need"),
        "project_need": section(text, "Project Need", "Project Status"),
        "project_type": None,
        "status": section(text, "Project Status", "Planned In-Service Date"),
        "start_date": None,
        "end_date": None,
        "in_service_date": date,
        "in_service_date_raw": date_raw,
        "in_service_dates": json.dumps(dates),
        "estimated_cost": total,
        "currency": "USD",
        "cost_basis": basis,
        **budget,
        "cost_total_difference": difference,
        "estimated_cost_raw": cost_raw,
        "location": None,
        "location_method": "unresolved",
        "source_url": URL,
        "source_file": PDF_PATH.name,
        "source_sha256": source_sha256,
        "source_page": page["source_page"],
        "source_project_number": int(header.group(1)),
        "source_project_count": int(header.group(2)),
        "quality_flags": "; ".join(sorted(set(issues))),
    }


records = []
for page in pages:
    try:
        records.append(parse_project(page))
    except ValueError as error:
        raise ValueError(f"Could not parse page {page['source_page']}: {error}") from error

df_dominion = pd.DataFrame(records)
for column in ["estimated_cost", *BUDGET_COLUMNS, "cost_total_difference"]:
    df_dominion[column] = df_dominion[column].astype("Int64")
for column in ["start_date", "end_date", "in_service_date"]:
    df_dominion[column] = pd.to_datetime(df_dominion[column], errors="raise")
display(df_dominion[["project_id", "project_name", "status", "in_service_date", "estimated_cost"]].head(10))

,project_id,project_name,status,in_service_date,estimated_cost
0,06810 H,Summerville 115kV Loop: Rebuild,In Progress,NaT,15775885
1,02726 K,Columbia Canal 115 kV: Rebuild,In Progress,2026-05-31,1150364
2,6853 B-F,Scout 230 kV Sub and Fold-in: Construct,In Progress,2026-05-31,19418000
3,6846 A,Eastover – Sumter 115kV DEP Tie: Rebuild with 1272 ACSR,In Progress,2026-05-31,1238443
4,6809 N,Batesburg - Saluda County 115kV: Rebuild,In Progress,NaT,13040765
5,06874 B,Summerville-Boone Hill 115kV: Move Line for US78 SCDOT Road Widening,In Progress,2026-07-31,12475000
6,6852,Urquhart – Toolebeck 115kV line: Rebuild,In Progress,2026-08-12,15948620
7,6859,Dawson 230kV Sub and Fold-in: Construct and Rebuild,In Progress,NaT,93603207
8,6341 A-F,Cainhoy - Hamlin 115kV: Rebuild Line and Cainhoy – Hamlin 115 kV #2: Construct New 115 kV Line,In Progress,2026-12-02,40858628
9,5392 A-C,Coit – Gills Creek 115kV: Construct,In Progress,2026-12-31,5773202


## 3. Validate completeness and review source issues
Structural failures stop the export. Source inconsistencies remain visible in `quality_flags` and a separate review CSV. Invalid dates stay missing; phased dates are preserved as a list and in the raw field.

The document does not provide structured coordinates, construction start/end dates, or a common project-type taxonomy. Leave these fields missing for later enrichment. A planned in-service date is not a construction interval.

Two distinct source projects share ID `6809 M`; retain both and flag them. Use `source_sha256` plus `source_page` to identify a source record uniquely.

In [4]:
assert not df_dominion.empty, "No projects extracted"
assert df_dominion["source_project_count"].nunique() == 1, "Inconsistent project counts"
expected_count = int(df_dominion["source_project_count"].iloc[0])
assert len(df_dominion) == expected_count == len(df_pages), "Missing or extra project pages"
assert sorted(df_dominion["source_project_number"].tolist()) == list(range(1, expected_count + 1))
duplicate_ids = df_dominion["project_id"].duplicated(keep=False)
for index in df_dominion.index[duplicate_ids]:
    flags = set(filter(None, df_dominion.at[index, "quality_flags"].split("; ")))
    flags.add("duplicate_project_id")
    df_dominion.at[index, "quality_flags"] = "; ".join(sorted(flags))
for column in ["project_id", "project_name", "description", "project_need", "status"]:
    assert df_dominion[column].notna().all() and df_dominion[column].str.strip().ne("").all(), column
assert df_dominion["estimated_cost"].notna().all()
assert df_dominion["estimated_cost"].ge(0).all()

review = df_dominion.loc[df_dominion["quality_flags"].ne("")].copy()
print(f"Validated {len(df_dominion)} projects; {len(review)} need source review.")
display(review[["source_page", "project_id", "in_service_date_raw", "cost_total_difference", "quality_flags"]])

Validated 54 projects; 15 need source review.


,source_page,project_id,in_service_date_raw,cost_total_difference,quality_flags
0,1,06810 H,04/31/26,0,invalid_in_service_date:04/31/26
2,3,6853 B-F,5/31/2026,-2590000,cost_total_mismatch
3,4,6846 A,5/31/2026,-850000,cost_total_mismatch
4,5,6809 N,06/31/2026,0,invalid_in_service_date:06/31/2026
7,8,6859,10/1/2025 (phase 1) and 10/1/2026 (phase 2),0,multiple_in_service_dates
11,12,06367 D - G,12/01/2026,0,nonstandard_cost_format
14,15,6870 A,12/31/2026,270000,cost_total_mismatch
15,16,6847,5/1/2027,-5617468,cost_total_mismatch
18,19,6809 M,12/31/2027,0,duplicate_project_id
24,25,6810 J,12/31/2027,100000,cost_total_mismatch


## 4. Explore the extracted portfolio
The total below combines the published budget totals and the two narrative estimates. It is a source summary, subject to the review flags above. The year summary excludes missing and phased dates.

In [5]:
print(f"Sum of source-reported estimates: ${df_dominion['estimated_cost'].sum():,}")
display(df_dominion.groupby("status", dropna=False).agg(
    projects=("project_id", "size"), estimated_cost_usd=("estimated_cost", "sum")
))
display(df_dominion.assign(in_service_year=df_dominion["in_service_date"].dt.year.astype("Int64"))
        .groupby("in_service_year", dropna=False)
        .agg(projects=("project_id", "size"), estimated_cost_usd=("estimated_cost", "sum")))
display(df_dominion.nlargest(10, "estimated_cost")[["project_name", "estimated_cost", "cost_basis", "quality_flags"]])

Sum of source-reported estimates: $892,741,934


,projects,estimated_cost_usd
status,,
In Progress,25,522461994
Planned,29,370279940


,projects,estimated_cost_usd
in_service_year,,
2026,12,193512035
2027,14,188078073
2028,14,140427977
2029,7,178314514
2030,2,41364478
2031,1,9750000
2032,1,18875000
<NA>,3,122419857


,project_name,estimated_cost,cost_basis,quality_flags
7,Dawson 230kV Sub and Fold-in: Construct and Rebuild,93603207,published_budget_total,multiple_in_service_dates
44,Wateree-Killian 230kV: Rebuild,44517719,published_budget_total,
45,Church Creek – Dawson 230 kV: Rebuild from Long Savannah to Dawson,42375000,published_budget_total,
12,Riverport 115kV Tap: Construct Tap,41389047,published_budget_total,
8,Cainhoy - Hamlin 115kV: Rebuild Line and Cainhoy – Hamlin 115 kV #2: Construct New 115 kV Line,40858628,published_budget_total,
16,Yemassee– Ritter 230kV #1 & #2: Construct SPDC with B-1272,39066742,published_budget_total,
43,Canadys-Ritter 115KV: Rebuild SPDC 230/115KV 1272 (Approx 18 Miles),38121795,published_budget_total,cost_total_mismatch
30,Williams-Summerville 230kV: Upgrade to SPDC B1272 ACSR,31100000,published_budget_total,cost_total_mismatch
29,Winnsboro West 230-115 kV Sub and Fold-in: Construct,23272000,customer_financed_estimate,
15,Church Creek – Faber Place – Charleston Transmission: Add 230kV Line,22788174,published_budget_total,cost_total_mismatch


## 5. Save processed data and provenance
Exports are rebuilt from the cached PDF on each run. `dominion_projects.csv` is the normalized table, `dominion_review.csv` contains flagged rows, and `dominion_pages.jsonl` preserves extracted page text. The manifest records the source checksum. Raw source bytes are left untouched.

When loading the CSV later, use `pd.read_csv(path, dtype={"project_id": "string"})` to retain ID formatting. Geocoding and cross-utility matching are later steps; this notebook does not invent geometry.

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
projects_path = OUTPUT_DIR / "dominion_projects.csv"
df_dominion.to_csv(projects_path, index=False, date_format="%Y-%m-%d")
review.to_csv(OUTPUT_DIR / "dominion_review.csv", index=False, date_format="%Y-%m-%d")
df_dominion.to_json(OUTPUT_DIR / "dominion_projects.json", orient="records", date_format="iso", indent=2)
df_pages.to_json(OUTPUT_DIR / "dominion_pages.jsonl", orient="records", lines=True, force_ascii=False)
manifest = {
    "source_url": URL,
    "source_file": str(PDF_PATH.relative_to(RAW_DATA_DIR)),
    "source_sha256": source_sha256,
    "page_count": len(df_pages),
    "project_count": len(df_dominion),
    "review_count": len(review),
    "cost_currency": "USD",
}
(OUTPUT_DIR / "dominion_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
reloaded = pd.read_csv(projects_path, dtype={"project_id": "string"})
assert reloaded["project_id"].tolist() == df_dominion["project_id"].tolist()
assert len(reloaded) == expected_count
assert hashlib.sha256(PDF_PATH.read_bytes()).hexdigest() == source_sha256
print(f"Saved {len(reloaded)} projects to {projects_path}")

Saved 54 projects to C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\dominion\dominion_projects.csv
